# Методологический цикл MTender — проход по артефактам

Ноутбук воспроизводит **ключевые таблицы** методологического цикла из сохранённых артефактов `results/`.
Сами расчёты (бутстрапы, 20 сидов, ablation-перефиты) выполняются скриптами — их запуск здесь был бы
бессмысленно медленным; см. команды в последней ячейке и `docs/model_card.md` §10.

Отчёт цикла: `docs/METHODOLOGY_CYCLE.md`. Аудит смещения: `reports/bias_audit.md`.

**Индикатор потенциального риска, а не доказательство коррупции.**

## Шаг 0 — baseline: голоса детекторов и полосы

Пересчёт из `results/tables/tender_anomaly_scores.csv`; должен совпасть с `detection_summary.json`
и `/api/stats`: low 125 242 / medium 19 474 / high 4 631 / critical 3 031.

In [ ]:
import pandas as pd

scores = pd.read_csv('results/tables/tender_anomaly_scores.csv',
                     usecols=['ocid', 'n_models_flagging', 'consensus', 'max_score'])
votes = scores['n_models_flagging'].fillna(0).astype(int)
dist = votes.value_counts().reindex(range(5)).fillna(0).astype(int)
bands = {'low (0)': dist[0], 'medium (1)': dist[1], 'high (2)': dist[2],
         'critical (>=3)': int((votes >= 3).sum())}
print('строк:', len(scores))
print('распределение голосов 0..4:', dist.to_dict())
print('полосы:', bands)
assert bands['critical (>=3)'] == 3031, 'baseline changed — investigate before quoting numbers'

## Шаг 2 — внешняя валидация (AAP)

18 exact-совпадений из 22 записей, 633 размеченных процедуры, base rate 0.415%.
Производственное ранжирование: ROC-AUC **0.493** (ДИ пересекает 0.5), precision@200 = 0.
Обогащение — только в хвосте консенсуса (lift ≈ 2–2.5).

In [ ]:
ext = pd.read_csv('results/tables/external_metrics.csv')
head = ext[(ext.period == 'all') &
           (ext.metric.isin(['roc_auc', 'pr_auc', 'precision@200', 'precision@1000']))]
print(head[['ranking_rule', 'metric', 'value', 'ci_lo', 'ci_hi']].to_string(index=False))

sweep = pd.read_csv('results/tables/threshold_sweep.csv')
rules = sweep[(sweep.period == 'all') &
              (sweep.rule.isin(['votes>=1', 'votes>=2', 'votes>=3', 'votes>=4',
                                'baseline:no_flag']))]
print()
print(rules[['rule', 'n_flagged', 'precision', 'lift',
              'cost_miss_10', 'cost_miss_50', 'cost_miss_200']].to_string(index=False))

## Шаг 3 — круговая логика и дубликат

- Колонки, которые детекторы **не видели**, тоже «значимы» против флага консенсуса → ANOVA меряет конструкцию.
- Плацебо-шум в детекторах значимости не набирает (p = 0.72) → тест не вакуумный.
- `winner_share_of_buyer_awards` ≡ `buyer_supplier_repeat_share` (100% строк).

In [ ]:
ctrl = pd.read_csv('results/tables/circularity_anova_controls.csv')
print('контрольные колонки (НЕ в детекторах):')
print(ctrl[~ctrl.fed_to_detectors].to_string(index=False))

contrib = pd.read_csv('results/tables/feature_contribution.csv')
print()
print('топ признаков по внешней дискриминации:')
print(contrib[['feature', 'univariate_auc_external', 'verdict']].head(8).to_string(index=False))
print()
print('дубликат-пара:')
print(contrib[contrib.feature.isin(['buyer_supplier_repeat_share',
                                     'winner_share_of_buyer_awards'])]
      [['feature', 'univariate_auc_external', 'verdict']].to_string(index=False))

## Шаги 4–7 — согласие, сиды, дрейф

- Средняя попарная kappa 0.250; 97.6% «хотя бы одного» детектора помечены не всеми.
- 20 сидов: количество консенсуса cv 4.8%, но Jaccard состава ≈ 0.71.
- Дрейф: PSI 0.29–0.41; доля консенсуса 4.3% (2018) → 0.69% (2024).

In [ ]:
import json

agreement = json.load(open('results/methodology/agreement_matrix.json'))
print('kappa: средний', round(agreement['mean_pairwise_kappa'], 3),
      '/ минимум', round(agreement['min_pairwise_kappa'], 3))
print('доля расхождений среди помеченных:',
      round(agreement['share_disagreement_among_flagged'], 3))

runs = pd.read_csv('results/tables/seed_stability_runs.csv')
print()
print('20 сидов: консенсус', round(runs.n_consensus_ge3.mean(), 1),
      '±', round(runs.n_consensus_ge3.std(), 1),
      '; Jaccard к seed42 средний', round(runs.jaccard_consensus_vs_seed42.mean(), 3))

drift = pd.read_csv('results/tables/drift_periods.csv')
print()
print(drift.to_string(index=False))

by_year = pd.read_csv('results/tables/drift_by_year.csv')
print()
print(by_year[['year', 'n', 'share_votes_ge_3', 'share_if_flag', 'n_positive']]
      .to_string(index=False))

## Раздел E — аудит смещения

41 группа; 16 нарушений / 13 объяснённых легитимными факторами / 8 допустимо / 4 предупреждения.
Главное: полосы измеряют размер контракта (≥50 млн: 50.9% в high+, disparity 10.5),
прямые закупки (51.5% корпуса) почти не флагуются, мелкие заказчики и «unknown»-регион — артефакты данных.

In [ ]:
groups = pd.read_csv('results/tables/bias_audit_groups.csv')
print(groups.verdict.value_counts().to_string())
print()
flagged = groups[groups.verdict.str.startswith(('нарушение', 'предупреждение'))]
print(flagged[['group_dimension', 'group', 'share_high_plus', 'disparity_ratio',
               'conditional_gap_pp', 'verdict']].head(12).to_string(index=False))

proxies = pd.read_csv('results/tables/bias_audit_feature_proxies.csv')
print()
print('признаки-прокси (|r| >= 0.3):')
print(proxies[proxies.flagged_structural_proxy][['feature', 'max_abs_corr']].to_string(index=False))

## Итерационный протокол — версии v0..v3

Одно изменение за цикл, единый инструмент метрик (`scripts/20`), решения по tuning 2018–2021. Ни одна версия не улучшила внешний AUC за пределами ДИ — шлифовка остановлена после 3 циклов. Сводка — `reports/cycle_dynamics.md`.

In [ ]:
import pandas as pd

cm = pd.read_csv('results/tables/cycle_metrics.csv')
cols = ['variant', 'n_features', 'consensus_n', 'auc_tuning', 'auc_holdout',
        'mean_kappa', 'top_n_jaccard_mean', 'stop_criteria_passed',
        'delta_auc_vs_v0_tuning']
print(cm[cols].to_string(index=False))
best = cm.stop_criteria_passed.max()
assert cm.loc[cm.variant == 'v0_baseline', 'consensus_n'].iloc[0] == 3031
assert best <= 3, 'no version should pass the convergence criteria'

## Проверка круговой логики — раздел B (`reports/circularity_check.md`)

Вердикт по каждому признаку: «подтверждён независимыми метками / только внутренне / не подтверждён». «Подтверждён» = подтверждённая внешняя ассоциация с AAP (в т.ч. обратная), а не валидность признака.

In [ ]:
import json
import pandas as pd

circ = json.load(open('results/methodology/circularity_check.json'))
s = circ['sections']
print('B1 ANOVA внешние vs полосы, Спирмен:', s['B1']['spearman_F_external_vs_F_bands'])
print('B2 половины: J(a,b) средний', s['B2']['jaccard_a_vs_b_mean'],
      '| детекторы', s['B2']['detector_pairwise_jaccard_mean'])
print('B4 permutation self vs external, Спирмен:',
      s['B4']['spearman_perm_self_vs_external'])
print('B5 после перетасовки связей: AUC', s['B5']['auc_external_shuffled'],
      '(был', s['B5']['auc_external_full'], ')')
verd = pd.read_csv('results/tables/circularity_verdicts.csv')
print(verd.verdict.value_counts().to_dict())
const = verd[verd.feature == 'accelerated_like'].iloc[0]
assert const.verdict.startswith('не подтверждён'), 'constant feature cannot be confirmed'

## Воспроизведение расчётов

```bash
.venv/bin/python scripts/12_external_metrics.py --boot 1000   # внешняя валидация
.venv/bin/python scripts/13_agreement_matrix.py               # согласие детекторов
.venv/bin/python scripts/14_threshold_calibration.py          # пороги, сплит, дрейф
.venv/bin/python scripts/15_robustness_seeds.py --runs 20     # устойчивость к сидам
.venv/bin/python scripts/16_feature_ablation.py               # вклад + ablation + круговая логика
.venv/bin/python scripts/17_explain_ocid.py --ocid <OCID>     # право на объяснение
.venv/bin/python scripts/18_bias_audit.py                     # аудит смещения
.venv/bin/python scripts/19_final_verification.py             # перепроверка всех чисел
.venv/bin/python scripts/20_cycle_framework.py --variants v0 v1 v2 v3  # итерационный протокол
.venv/bin/python scripts/21_circularity_check.py              # проверка круговой логики (раздел B)
.venv/bin/python scripts/22_cycle_dynamics.py                 # динамика метрик по циклам
```

**Индикатор потенциального риска, а не доказательство коррупции.**